In [93]:
from bs4 import BeautifulSoup as soup
from bs4 import Tag, NavigableString
import mwparserfromhell
import requests
import pandas
import re

In [94]:
# Test code for downloaded file wiki-'eye'.html
# request page for eye online

with open("wiki-'eye'.html", 'r', encoding='utf-8') as f:
    page = f.read()
soupify = soup(page, 'html.parser')
# wikicode = mwparserfromhell.parse(soupify)
# mw-heading 2 for language, 
#   mw-heading 3 for subheadings like etymology, pronunciation, etc. (look at next element: paragraph, list, etc.)
    # span class="IPA nowrap" for pronunciation?
        # pronunciation can be mw-heading 4???
            # mayhaps id contains "Pronunciation"?, then look for list element
            # thus, id contains "Etymology" for etymology section?
    # span class="etyl" for etymology links (Ole English, Middle English)
    # i class="Latn" for Latin script (for non-English words), mention class in the etymology section
    # WHY ISN'T THE WIKIMEDIA PAGE NESTED AT ALL!!!?!?!!    
# wikicode.filter_templates()
soupify.find_all('h3') # seemingly gets relevant headings for this page

# feels a bit specific but worth a shot
soupify.find_all('h3')[0].parent.next_sibling.next_sibling.next_element.find('span', class_='IPA').text # get IPA pronuncation from first h3 heading (pronunciation) and next sibling (list of pronunciations)'

# okay, let's try parsing the etymology section, which is the second h3 heading
all_etyl = soupify.find_all('h3')[1].parent.next_sibling.next_sibling.find_all('i', class_='Latn')
[x.text for x in all_etyl]

# unfortunately, we cannot get the related term "ogle" at all - no ids or anything

['eye', 'yë', 'eyghe', 'ēage', '*augā', '*augô', '*h₃okʷ-', '*h₃ekʷ-']

In [206]:
def get_etym_links(etym_p): 
    # etym_links structure: {language: [{"word": word, "link": link}, ...], ...}
    etym_links = {"Other Tag" : [], "Unknown Language": []}
    language = None
    for element in etym_p.descendants:
        print("Element: ", element)
        if isinstance(element, NavigableString): 
            continue
        # if element is a "language" link, establish language key in dict 
        elif element.name == "span" and element.get("class") and "etyl" in element.get("class"):
            language = element.text.replace(" ", "_")
            if language not in etym_links:
                etym_links[language] = []

        # if element is a latin-script word, add to dict with language key if possible, otherwise add to "Unknown" key; if it's a link, add the link as well
        elif element.name == "i" and "Latn" in element.get("class"):
            link = None
            temp = element.next_sibling
            while temp and not link:
                temp = temp.next_sibling
                if isinstance(temp, Tag):
                    link = temp
            print("Element: ", element, "Link: ", link)
            word_dict = {"word": element.text, "link" : link.get("href") if link and link.name == "a" else None}
            
            if language:
                etym_links[language].append(word_dict)
            
            else:
                etym_links["Unknown Language"].append(word_dict)
        # if element is a link but we don't have a language established, add to "Other" key with link if possible
        elif element.name == "a" and language:
            word_dict = {"word": element.text, "link" : element.get("href")}
            etym_links["Other Tag"].append(word_dict)
        
    return etym_links

In [136]:
def get_pronunciation(pron_ul): 
    # for li in pron_ul.find_all("li"):
        # if li.find("span", class_="IPA nowrap"):
        #     return li.find("span", class_="IPA nowrap").text
        return pron_ul.find("span", class_="IPA nowrap").text if pron_ul.find("span", class_="IPA nowrap") else None

In [137]:
def parse_etym_links(etym_links): 
    return

In [151]:
def filter_language_sections(page_soup, language): 
    headers = page_soup.find_all("h2")
    # 
    language_section_idx = headers.index(page_soup.find("h2", id=language))
    language_section = headers[language_section_idx]

    relevant_html = []    
    for html in language_section.find_all_next():
        if html.name == "h2" and html != language_section:
            break
        if isinstance(html, Tag):
            relevant_html.append(html)    
            
    return soup("".join(str(html) for html in relevant_html), "html.parser")

In [ ]:
# Scrapes Wiktionary page for relevant links, etymology, pronunciation, compacts into pandas df
def scrape_wiktionary_page(html_file):
    
    # url = f"https://en.wiktionary.org/wiki/{word}"
    # page = requests.get(url)
    # print(html_file)
    if html_file.startswith("http"):
        grabbed_html = grab_page_html(html_file)
        html_content = grabbed_html#.decode("utf-8")
        word = html_file.split("/")[-1].split(".")[0] # get word from file name
    else:
        with open(html_file, "r", encoding="utf-8") as f:
            html_content = f.read()
        word = html_file.split(".")[0].split("-")[-1] # get word from file name
    page_soup = soup(html_content, "html.parser")

    # print(page_soup)
    relevant_html = filter_language_sections(page_soup, "English")
    # print("Relevant html: ", relevant_html)

    # grab all etymology sections that have id containing "Etymology" that are in the english-language section (some words have multiple etymologies)
    etymology = relevant_html.find_all("h3", id=re.compile("Etymology"))
    etym_dict = {}
    if etymology:
        for etym in etymology:
            if etym.find_next("p"):
                etym_dict[etym.text] = get_etym_links(etym.find_next("p"))

    # Get pronunciation
    pronunciation = relevant_html.find("h3", id="Pronunciation")

    if pronunciation:
        pronunciation_ul = pronunciation.find_next("ul")
        pronunciation_text = get_pronunciation(pronunciation_ul)
    else:
        pronunciation_text = None

    # # Get related links (e.g., synonyms, antonyms, related terms)
    # related_links = []
    # for section in page_soup.find_all("h3", class_="mw-headline"):
    #     if section.text in ["Synonyms", "Antonyms", "Related terms" ]:
    #         links = section.find_next("ul").find_all("a")
    #         related_links.extend([link.text for link in links])

    # Compile into pandas DataFrame
    data = {
        "Word": word,
        "Etymology": etym_dict,
        "Pronunciation": pronunciation_text,
        # "Related Links": related_links
        # "Further Enquiry" : 
    }
    
    df = pandas.DataFrame([data])
    return df
    
def grab_page_html(word):
    url = f"https://en.wiktionary.org/wiki/{word}"
    page = requests.get(url)
    return page.content

In [207]:
df = scrape_wiktionary_page("wiki-'eye'.html")
df

wiki-'eye'.html
Element:  From 
Element:  <span class="etyl"><a class="extiw" href="https://en.wikipedia.org/wiki/Middle_English" title="w:Middle English">Middle English</a></span>
Element:  <a class="extiw" href="https://en.wikipedia.org/wiki/Middle_English" title="w:Middle English">Middle English</a>
Element:  Middle English
Element:   
Element:  <i class="Latn mention" lang="enm"><a class="mw-selflink-fragment" href="#Middle_English:_eye">eye</a></i>
Element:  <i class="Latn mention" lang="enm"><a class="mw-selflink-fragment" href="#Middle_English:_eye">eye</a></i> Link:  <i class="Latn mention" lang="enm"><a href="/wiki/ye#Middle_English" title="ye">yë</a></i>
Element:  <a class="mw-selflink-fragment" href="#Middle_English:_eye">eye</a>
Element:  eye
Element:  , 
Element:  <i class="Latn mention" lang="enm"><a href="/wiki/ye#Middle_English" title="ye">yë</a></i>
Element:  <i class="Latn mention" lang="enm"><a href="/wiki/ye#Middle_English" title="ye">yë</a></i> Link:  <i class="Lat

TypeError: argument of type 'NoneType' is not iterable

In [190]:
# 200 ihops is one really good tattoo btw 
# df["Etymology"][0]
# for item in df["Etymology"][0]:
    # print(item)
    # for x in item:
    #     print(x, item[x])
for key, val in df["Etymology"][0]["Etymology 1"].items():
    for item in val:
        print(key, item)

# df["Etymology"][0]

Other {'word': 'Middle English', 'link': 'https://en.wikipedia.org/wiki/Middle_English'}
Other {'word': 'eye', 'link': '#Middle_English:_eye'}
Other {'word': 'yë', 'link': '/wiki/ye#Middle_English'}
Other {'word': 'eyghe', 'link': '/wiki/eyghe#Middle_English'}
Other {'word': 'Old English', 'link': 'https://en.wikipedia.org/wiki/Old_English'}
Other {'word': 'ēage', 'link': '/wiki/eage#Old_English'}
Other {'word': 'Proto-West Germanic', 'link': 'https://en.wikipedia.org/wiki/Proto-West_Germanic_language'}
Other {'word': '*augā', 'link': '/wiki/Reconstruction:Proto-West_Germanic/aug%C4%81'}
Other {'word': 'Proto-Germanic', 'link': 'https://en.wikipedia.org/wiki/Proto-Germanic_language'}
Other {'word': '*augô', 'link': '/wiki/Reconstruction:Proto-Germanic/aug%C3%B4'}
Other {'word': 'Proto-Indo-European', 'link': 'https://en.wikipedia.org/wiki/Proto-Indo-European_language'}
Other {'word': '*h₃okʷ-', 'link': '/wiki/Reconstruction:Proto-Indo-European/h%E2%82%83ok%CA%B7-'}
Other {'word': '*h₃e